In [3]:
from data.ingest.portwatch.run import table_name, data_path
from data.ingest.utilities import lake_connection
con = lake_connection(data_path=data_path)
df = con.execute(
    "SELECT * " \
    f"FROM {table_name}").fetchdf()


In [ ]:
con.execute(f"DROP TABLE IF EXISTS {table_name}")

,portid,portname,country,ISO3,continent,fullname,lat,lon,vessel_count_total,vessel_count_container,...,vessel_count_tanker,industry_top1,industry_top2,industry_top3,share_country_maritime_import,share_country_maritime_export,LOCODE,pageid,countrynoaccents,ObjectId
0,port1325,Tsuruga,Japan,JPN,Asia & Pacific,"Tsuruga, Japan",35.667194,136.070611,529,77,...,54,Mineral Products,Wood & Wood Products,Vegetable Products,0.42,0.09,NaN,3b1f40eb8acd452ab6229331302891fe,Japan,1
1,port1198,Sibolga,Indonesia,IDN,Asia & Pacific,"Sibolga, Indonesia",1.734969,98.781715,81,28,...,39,Mineral Products,Prepared Foodstuffs & Beverages,Vegetable Products,0.08,0.01,NaN,4d00b008b4be4068bf8a19d7da6e8cf6,Indonesia,2
2,port339,Fangcheng,China,CHN,Asia & Pacific,"Fangcheng, China",21.614205,108.364339,1968,34,...,211,Mineral Products,Chemical & Allied Industries,Vegetable Products,1.90,0.42,NaN,266042b244cf452a89f4a2de14f3da2f,China,3
3,port1335,Ube,Japan,JPN,Asia & Pacific,"Ube, Japan",33.944875,131.213184,4799,13,...,2089,Mineral Products,Chemical & Allied Industries,"Plastics, Rubber, Leather",0.76,2.35,NaN,f3f6725ef00e4d3f8f9c93c45258667e,Japan,4
4,port153,Bluff Harbor,New Zealand,NZL,Asia & Pacific,"Bluff Harbor, New Zealand",-46.590130,168.360216,240,43,...,58,Wood & Wood Products,Metals,Prepared Foodstuffs & Beverages,7.79,2.84,NaN,894ab5e5a18e48b6922c023196f74c8f,New Zealand,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,port1279,Teesport,United Kingdom,GBR,Europe,"Teesport, United Kingdom",54.603718,-1.149997,2607,628,...,1163,Mineral Products,Chemical & Allied Industries,Vegetable Products,3.85,12.41,GB TEE,5786371d71d34b5d898171cf2cea9bc2,United Kingdom,997
996,port481,Houston,United States,USA,North America,"Houston, United States",29.720231,-95.139788,7358,984,...,5014,Mineral Products,Chemical & Allied Industries,Vegetable Products,8.21,17.78,US HOU,1581e7affcc84016af4e4bbf372e721b,United States,998
997,port1289,Thamesport,United Kingdom,GBR,Europe,"Thamesport, United Kingdom",51.437876,0.685666,421,240,...,110,Mineral Products,Chemical & Allied Industries,"Plastics, Rubber, Leather",2.89,0.04,GB THP,b59c0cb18ebd420dbf6318b515bb20e5,United Kingdom,999
998,port1298,Tilbury,United Kingdom,GBR,Europe,"Tilbury, United Kingdom",51.458526,0.342468,1558,785,...,92,Mineral Products,Vegetable Products,Wood & Wood Products,2.31,1.25,GB TIL,50d8deda320947ed8f2c0e69508f42af,United Kingdom,1000


In [3]:
df.loc[0]

portid                                                   port1325
portname                                                  Tsuruga
country                                                     Japan
ISO3                                                          JPN
continent                                          Asia & Pacific
fullname                                           Tsuruga, Japan
lat                                                     35.667194
lon                                                    136.070611
vessel_count_total                                            529
vessel_count_container                                         77
vessel_count_dry_bulk                                          96
vessel_count_general_cargo                                    115
vessel_count_RoRo                                             187
vessel_count_tanker                                            54
industry_top1                                    Mineral Products
industry_t

In [ ]:
con.execute("SELECT * FROM lake.snapshots();").fetch_df()

,snapshot_id,snapshot_time,schema_version,changes,author,commit_message,commit_extra_info
0,0,2026-04-03 18:28:55.480580+02:00,0,{'schemas_created': ['main']},None,None,None
1,1,2026-04-03 18:28:55.541765+02:00,1,"{'tables_created': ['main.weather'], 'tables_i...",None,None,None


In [1]:
import duckdb
con = duckdb.connect()
con.execute("ATTACH 'ducklake:sqlite:data/metadata/metadata.sqlite' AS lake (DATA_PATH 'data/parquet/')")
con.execute("USE lake")

In [6]:
con.execute(
    # "CREATE TABLE people (id INTEGER, name VARCHAR, salary FLOAT);"
    "INSERT INTO people VALUES (3, 'John', 92_000.0), (4, 'Anna', 100_000.0);"
    )

In [ ]:
con.execute("""
                MERGE INTO people
                USING (
                    SELECT
                        unnest([3, 1]) AS id,
                        unnest(['Sarah', 'John']) AS name,
                        unnest([95_000.0, 105_000.0]) AS salary
                ) AS upserts
                ON (upserts.id = people.id)
                WHEN MATCHED THEN UPDATE
                WHEN NOT MATCHED THEN INSERT;

            FROM people;""")

In [7]:
con.execute("SELECT * FROM lake.snapshots();").fetchdf()

,snapshot_id,snapshot_time,schema_version,changes,author,commit_message,commit_extra_info
0,0,2026-05-24 18:08:26.669465+02:00,0,{'schemas_created': ['main']},None,None,None
1,1,2026-05-24 18:08:29.581624+02:00,1,{'tables_created': ['main.people']},None,None,None
2,2,2026-05-24 18:08:29.593098+02:00,1,{'inlined_insert': ['1']},None,None,None
3,3,2026-05-24 18:08:31.921421+02:00,1,"{'tables_inserted_into': ['1'], 'inlined_delet...",None,None,None
4,4,2026-05-24 18:09:28.013817+02:00,1,{'inlined_insert': ['1']},None,None,None
5,5,2026-05-24 18:09:58.976277+02:00,1,{'inlined_insert': ['1']},None,None,None


In [8]:
con.execute("SELECT * FROM people;").fetchdf()

,id,name,salary
0,1,John,105000.0
1,3,Sarah,95000.0
2,2,Anna,100000.0
3,3,John,92000.0
4,4,Anna,100000.0
5,3,John,92000.0
6,4,Anna,100000.0
